# K-Means Clustering on Retail Customer Data

**Objective:** Segment retail customers into groups based on purchasing behavior using K-Means clustering.

**Dataset:** Synthetic retail customer transactions (income, spending, purchase frequency).

**Pipeline:** Load Data → EDA → Preprocessing → Elbow/Silhouette → K-Means → Cluster Analysis

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

from retail_data import generate_retail_dataset

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42

## 1. Load Retail Dataset

In [ ]:
df = generate_retail_dataset(n_samples=2000, random_state=RANDOM_STATE)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
numerical_cols = ['Age', 'Annual_Income', 'Spending_Score', 'Num_Purchases', 'Avg_Transaction_Value', 'Total_Sales']
df[numerical_cols].hist(bins=30, figsize=(14, 8), edgecolor='black')
plt.suptitle('Distribution of Numerical Retail Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df[['Annual_Income', 'Spending_Score', 'Num_Purchases', 'Total_Sales']].sample(500, random_state=RANDOM_STATE),
             diag_kind='kde', corner=True)
plt.suptitle('Pairwise Relationships (Sample)', y=1.02)
plt.show()

In [ ]:
corr = df[numerical_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

## 3. Feature Selection & Preprocessing

K-Means works on numerical features. We select behavioral and demographic variables and scale them.

In [ ]:
feature_cols = ['Age', 'Annual_Income', 'Spending_Score', 'Num_Purchases', 'Avg_Transaction_Value']
X = df[feature_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Features used: {feature_cols}')
print(f'Scaled shape: {X_scaled.shape}')

## 4. Find Optimal K (Elbow & Silhouette)

In [ ]:
k_range = range(2, 11)
inertias, silhouettes = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(k_range, inertias, 'bo-')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(k_range, silhouettes, 'go-')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by K')
plt.tight_layout()
plt.show()

optimal_k = k_range[np.argmax(silhouettes)]
print(f'Optimal K (by silhouette): {optimal_k}')

## 5. Train Final K-Means Model

In [ ]:
OPTIMAL_K = optimal_k
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print('Cluster distribution:')
print(df['Cluster'].value_counts().sort_index())
print(f'\nSilhouette Score: {silhouette_score(X_scaled, df["Cluster"]):.4f}')
print(f'Davies-Bouldin Index: {davies_bouldin_score(X_scaled, df["Cluster"]):.4f}')
print(f'Calinski-Harabasz Score: {calinski_harabasz_score(X_scaled, df["Cluster"]):.4f}')

## 6. Visualize Clusters

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Annual_Income', y='Spending_Score', hue='Cluster', palette='Set2', s=60, alpha=0.7)
plt.title(f'K-Means Customer Segments (K={OPTIMAL_K})')
plt.xlabel('Annual Income ($)')
plt.ylabel('Spending Score (1-100)')
plt.legend(title='Cluster')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Num_Purchases', y='Total_Sales', hue='Cluster', palette='Set2', s=60, alpha=0.7)
plt.title('Clusters by Purchase Frequency vs Total Sales')
plt.show()

## 7. Cluster Profiling

In [ ]:
cluster_profile = df.groupby('Cluster')[feature_cols + ['Total_Sales']].mean().round(2)
cluster_profile

In [ ]:
segment_crosstab = pd.crosstab(df['Cluster'], df['Customer_Segment'], normalize='index').round(3)
print('Cluster vs Actual Customer Segment (proportion):')
segment_crosstab

## 8. Business Insights & Conclusions

- **K-Means** partitions customers into homogeneous groups based on numerical purchasing behavior.
- Use cluster profiles to design targeted marketing (e.g., high-income/low-spending = "savings-focused" campaigns).
- Compare discovered clusters with existing `Customer_Segment` labels to validate or refine segmentation strategy.
- **Limitation:** K-Means assumes spherical clusters and is sensitive to feature scaling — always standardize numerical features.